In [2]:
"""
HEDONIC PRICING MODEL v4: Критический пересмотр признаков
==========================================================
Цель: Проверить логическую обоснованность каждого признака перед включением в модель

Ключевые вопросы:
1. Есть ли мультиколлинеарность площадь ↔ комнатность?
2. Застройщик: прямое влияние или proxy для других факторов?
3. Этаж: как правильно моделировать (линейно, категориально)?
4. Уступка/ипотека: влияют ли на цену за м²?
5. Минимальная комнатность: корректное сравнение с 2-3к
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from scipy import stats

plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

print("="*80)
print("🔍 HEDONIC PRICING MODEL v4: КРИТИЧЕСКИЙ АНАЛИЗ ПРИЗНАКОВ")
print("="*80)

# =============================================================================
# 1. ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ
# =============================================================================
print("\n" + "="*60)
print("1. ЗАГРУЗКА ДАННЫХ")
print("="*60)

deals = pd.read_csv('clear_deals.csv')
projects = pd.read_csv('clear_projects.csv')

df = deals.merge(
    projects.drop_duplicates(subset="id_корпуса", keep="first")[
        ["id_корпуса", "проект", "класс_проекта", "девелопер", "этажей_от", "этажей_до"]
    ],
    on="id_корпуса", how="left"
)

# Корректировка на количество сделок
df['кол_во_квартир'] = df['суммарное_количество_сделок'].fillna(1).astype(int)
df['площадь_квартиры'] = df['суммарная_площадь_сделок'] / df['кол_во_квартир']
df['реальная_цена_квартиры'] = df['реальная_сумма_бюджета'] / df['кол_во_квартир']
df['цена_за_кв_м'] = df['реальная_цена_квартиры'] / df['площадь_квартиры']

# Класс жилья
df['класс_жилья'] = df['класс'].fillna(df['класс_проекта']).fillna('Не указан')

# Комнатность
def normalize_rooms(value):
    value = str(value).strip().lower()
    mapping = {
        'ст': 'Студия', 'студия': 'Студия', '0': 'Студия',
        '1': '1-комн', '2': '2-комн', '3': '3-комн', 
        '4': '4-комн', '5': '5+ комн', '6': '5+ комн'
    }
    return mapping.get(value, 'Другое')

df['тип_квартиры'] = df['количество_комнат'].astype(str).apply(normalize_rooms)

# Этаж
df['этаж_лота'] = pd.to_numeric(df['этаж_лота'], errors='coerce')
df['этажей_до'] = pd.to_numeric(df['этажей_до'], errors='coerce')
df['этажей_от'] = pd.to_numeric(df['этажей_от'], errors='coerce')

print(f"Загружено: {len(df):,} записей")

# =============================================================================
# 2. ФИЛЬТРАЦИЯ (убираем Бизнес, 4-комн, 5+)
# =============================================================================
print("\n" + "="*60)
print("2. ФИЛЬТРАЦИЯ ДАННЫХ")
print("="*60)

print(f"Исходно: {len(df):,}")

# Убираем класс "Бизнес"
df = df[df['класс_жилья'] != 'Бизнес']
print(f"После удаления Бизнес: {len(df):,}")

# Убираем 4-комн и 5+
df = df[~df['тип_квартиры'].isin(['4-комн', '5+ комн', 'Другое'])]
print(f"После удаления 4-комн и 5+: {len(df):,}")

# Убираем неуказанные классы
df = df[df['класс_жилья'] != 'Не указан']
print(f"После удаления неуказанных: {len(df):,}")

# Структура
print("\nФинальная структура:")
print(pd.crosstab(df['класс_жилья'], df['тип_квартиры']).to_string())

# =============================================================================
# 3. АНАЛИЗ МУЛЬТИКОЛЛИНЕАРНОСТИ: ПЛОЩАДЬ ↔ КОМНАТНОСТЬ
# =============================================================================
print("\n" + "="*60)
print("3. АНАЛИЗ: ПЛОЩАДЬ ↔ КОМНАТНОСТЬ (мультиколлинеарность)")
print("="*60)

fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# 3.1 Boxplot: площадь по типу квартиры
ax = axes[0, 0]
room_order = ['Студия', '1-комн', '2-комн', '3-комн']
df_plot = df[df['тип_квартиры'].isin(room_order)]
sns.boxplot(data=df_plot, x='тип_квартиры', y='площадь_квартиры', order=room_order, ax=ax)
ax.set_xlabel('Тип квартиры')
ax.set_ylabel('Площадь (м²)')
ax.set_title('Площадь по типу квартиры\n(высокая корреляция = мультиколлинеарность)')

# Средние значения
for i, room in enumerate(room_order):
    mean_area = df[df['тип_квартиры'] == room]['площадь_квартиры'].mean()
    ax.annotate(f'{mean_area:.0f}м²', xy=(i, mean_area), ha='center', fontweight='bold')

# 3.2 Корреляция площадь vs комнатность (числовая)
ax = axes[0, 1]
df_temp = df.copy()
room_to_num = {'Студия': 0, '1-комн': 1, '2-комн': 2, '3-комн': 3}
df_temp['комнат_число'] = df_temp['тип_квартиры'].map(room_to_num)
df_temp = df_temp.dropna(subset=['комнат_число', 'площадь_квартиры'])

corr = df_temp['комнат_число'].corr(df_temp['площадь_квартиры'])
ax.scatter(df_temp['комнат_число'], df_temp['площадь_квартиры'], alpha=0.1, s=5)
ax.set_xlabel('Количество комнат')
ax.set_ylabel('Площадь (м²)')
ax.set_title(f'Корреляция: r = {corr:.3f}\n(> 0.7 = высокая мультиколлинеарность)')
ax.set_xticks([0, 1, 2, 3])
ax.set_xticklabels(['Студия', '1-комн', '2-комн', '3-комн'])

# 3.3 Цена за м² vs площадь внутри каждого типа
ax = axes[1, 0]
colors = {'Студия': 'red', '1-комн': 'orange', '2-комн': 'blue', '3-комн': 'green'}
for room in room_order:
    room_data = df[df['тип_квартиры'] == room].sample(min(1000, len(df[df['тип_квартиры'] == room])), random_state=42)
    ax.scatter(room_data['площадь_квартиры'], room_data['цена_за_кв_м'], 
               alpha=0.3, s=10, label=room, color=colors[room])
ax.set_xlabel('Площадь (м²)')
ax.set_ylabel('Цена за м² (₽)')
ax.set_title('Цена за м² vs Площадь (по типам)\n(перекрытие = признаки дублируют информацию)')
ax.legend()

# 3.4 Распределение площади по типам (density)
ax = axes[1, 1]
for room in room_order:
    room_data = df[df['тип_квартиры'] == room]['площадь_квартиры']
    room_data.plot.kde(ax=ax, label=room, linewidth=2)
ax.set_xlabel('Площадь (м²)')
ax.set_title('Распределение площади по типам\n(перекрытие = сложно разделить эффекты)')
ax.legend()
ax.set_xlim(0, 150)

plt.tight_layout()
plt.savefig('01_multicollinearity_area_rooms.png', dpi=150, bbox_inches='tight')
plt.close()

print(f"\n✓ Корреляция площадь ↔ комнатность: r = {corr:.3f}")
if corr > 0.7:
    print("  ⚠️ ВЫСОКАЯ мультиколлинеарность! Нельзя включать оба признака.")
    print("  → РЕШЕНИЕ: Использовать только комнатность (тип_квартиры)")
    print("     Площадь уже заложена в комнатности.")
else:
    print("  ✓ Умеренная корреляция, но всё равно есть риск")

# =============================================================================
# 4. АНАЛИЗ: ЗАСТРОЙЩИК КАК CONFOUNDING VARIABLE
# =============================================================================
print("\n" + "="*60)
print("4. АНАЛИЗ: ЗАСТРОЙЩИК — прямое влияние или proxy?")
print("="*60)

fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# 4.1 Застройщик vs Класс
ax = axes[0, 0]
crosstab_dev_class = pd.crosstab(df['застройщик'], df['класс_жилья'], normalize='index') * 100
top_devs = df['застройщик'].value_counts().head(8).index
crosstab_dev_class.loc[top_devs].plot(kind='barh', stacked=True, ax=ax, colormap='viridis')
ax.set_xlabel('Доля (%)')
ax.set_ylabel('Застройщик')
ax.set_title('Застройщик vs Класс жилья\n(если застройщик строит только 1 класс → это proxy для класса)')
ax.legend(title='Класс')

# 4.2 Застройщик vs Район
ax = axes[0, 1]
crosstab_dev_district = pd.crosstab(df['застройщик'], df['район'], normalize='index') * 100
crosstab_dev_district.loc[top_devs].plot(kind='barh', stacked=True, ax=ax, colormap='tab10')
ax.set_xlabel('Доля (%)')
ax.set_ylabel('Застройщик')
ax.set_title('Застройщик vs Район\n(если застройщик работает только в 1 районе → proxy для локации)')
ax.legend(title='Район', fontsize=8)

# 4.3 Цена за м² по застройщикам (контролируя класс)
ax = axes[1, 0]
# Только Комфорт (чтобы убрать влияние класса)
comfort_only = df[df['класс_жилья'] == 'Комфорт']
dev_prices = comfort_only.groupby('застройщик')['цена_за_кв_м'].agg(['mean', 'std', 'count'])
dev_prices = dev_prices[dev_prices['count'] >= 100].sort_values('mean', ascending=True)
ax.barh(dev_prices.index, dev_prices['mean'], xerr=dev_prices['std']/np.sqrt(dev_prices['count'])*1.96)
ax.set_xlabel('Средняя цена за м² (₽)')
ax.set_title('Цена по застройщикам (только класс Комфорт)\n(разница = влияние застройщика ИЛИ района?)')

# 4.4 Застройщик внутри одного района
ax = axes[1, 1]
# Берём самый популярный район
top_district = df['район'].value_counts().index[0]
district_data = df[df['район'] == top_district]
dev_in_district = district_data.groupby('застройщик')['цена_за_кв_м'].agg(['mean', 'count'])
dev_in_district = dev_in_district[dev_in_district['count'] >= 50].sort_values('mean', ascending=True)
ax.barh(dev_in_district.index, dev_in_district['mean'])
ax.set_xlabel('Средняя цена за м² (₽)')
ax.set_title(f'Цена по застройщикам в районе "{top_district}"\n(если разница есть → застройщик влияет)')

plt.tight_layout()
plt.savefig('02_developer_confounding.png', dpi=150, bbox_inches='tight')
plt.close()

# Статистика
print("\nАнализ застройщиков:")
for dev in top_devs[:5]:
    dev_data = df[df['застройщик'] == dev]
    classes = dev_data['класс_жилья'].value_counts(normalize=True)
    districts = dev_data['район'].value_counts(normalize=True)
    print(f"\n{dev}:")
    print(f"  Классы: {dict(classes.round(2))}")
    print(f"  Районы: {dict(districts.head(3).round(2))}")

print("\n⚠️ ВЫВОД: Застройщик может быть proxy для:")
print("   - Класса жилья (некоторые строят только Комфорт)")
print("   - Района (некоторые работают только в 1-2 районах)")
print("   → РЕШЕНИЕ: Исключить застройщика из модели ИЛИ проверить VIF")

# =============================================================================
# 5. АНАЛИЗ: ЭТАЖ — как моделировать?
# =============================================================================
print("\n" + "="*60)
print("5. АНАЛИЗ: ЭТАЖ — линейно или категориально?")
print("="*60)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# 5.1 Цена vs Этаж (scatterplot)
ax = axes[0, 0]
sample = df.dropna(subset=['этаж_лота', 'цена_за_кв_м']).sample(min(5000, len(df)), random_state=42)
ax.scatter(sample['этаж_лота'], sample['цена_за_кв_м'], alpha=0.2, s=5)
# Тренд
z = np.polyfit(sample['этаж_лота'], sample['цена_за_кв_м'], 1)
x_line = np.linspace(1, sample['этаж_лота'].max(), 100)
ax.plot(x_line, np.poly1d(z)(x_line), 'r-', linewidth=2, label='Линейный тренд')
ax.set_xlabel('Этаж')
ax.set_ylabel('Цена за м² (₽)')
ax.set_title('Цена vs Этаж (линейный тренд)')
ax.legend()

# 5.2 Binned анализ этажа
ax = axes[0, 1]
df_temp = df.dropna(subset=['этаж_лота', 'цена_за_кв_м'])
df_temp['этаж_категория'] = pd.cut(df_temp['этаж_лота'], bins=[0, 1, 2, 3, 5, 10, 15, 20, 100],
                                   labels=['1', '2', '3', '4-5', '6-10', '11-15', '16-20', '20+'])
binned = df_temp.groupby('этаж_категория', observed=True)['цена_за_кв_м'].agg(['mean', 'std', 'count'])
ax.bar(range(len(binned)), binned['mean'], yerr=binned['std']/np.sqrt(binned['count'])*1.96, capsize=3)
ax.set_xticks(range(len(binned)))
ax.set_xticklabels(binned.index)
ax.set_xlabel('Категория этажа')
ax.set_ylabel('Средняя цена за м² (₽)')
ax.set_title('Binned анализ: Цена по категориям этажа\n(проверка нелинейности)')

# 5.3 Первый этаж vs остальные
ax = axes[0, 2]
first_floor = df[df['этаж_лота'] == 1]['цена_за_кв_м']
other_floors = df[df['этаж_лота'] > 1]['цена_за_кв_м']
ax.boxplot([first_floor, other_floors], labels=['1 этаж', 'Остальные'])
ax.set_ylabel('Цена за м² (₽)')
diff = other_floors.mean() - first_floor.mean()
ax.set_title(f'Первый этаж vs остальные\nРазница: {diff:+,.0f} ₽/м²')

# 5.4 Анализ "нижних" этажей (1-2 vs остальные)
ax = axes[1, 0]
# Проверяем, есть ли дома где продажи начинаются со 2 этажа
df_temp = df.dropna(subset=['этаж_лота', 'этажей_от'])
df_temp['первый_этаж'] = (df_temp['этаж_лота'] <= df_temp['этажей_от']).astype(int)
lower_floor = df_temp[df_temp['первый_этаж'] == 1]['цена_за_кв_м']
other = df_temp[df_temp['первый_этаж'] == 0]['цена_за_кв_м']
ax.boxplot([lower_floor, other], labels=['Нижний этаж\n(≤ этажей_от)', 'Остальные'])
ax.set_ylabel('Цена за м² (₽)')
diff_lower = other.mean() - lower_floor.mean()
ax.set_title(f'Нижний этаж (с учётом этажей_от) vs остальные\nРазница: {diff_lower:+,.0f} ₽/м²')

# 5.5 Последний этаж
ax = axes[1, 1]
df_temp = df.dropna(subset=['этаж_лота', 'этажей_до'])
df_temp['последний_этаж'] = (df_temp['этаж_лота'] == df_temp['этажей_до']).astype(int)
last_floor = df_temp[df_temp['последний_этаж'] == 1]['цена_за_кв_м']
other = df_temp[df_temp['последний_этаж'] == 0]['цена_за_кв_м']
ax.boxplot([last_floor, other], labels=['Последний этаж', 'Остальные'])
ax.set_ylabel('Цена за м² (₽)')
diff_last = last_floor.mean() - other.mean()
ax.set_title(f'Последний этаж vs остальные\nРазница: {diff_last:+,.0f} ₽/м²')

# 5.6 Относительный этаж
ax = axes[1, 2]
df_temp = df.dropna(subset=['этаж_лота', 'этажей_до', 'цена_за_кв_м'])
df_temp['относит_этаж'] = df_temp['этаж_лота'] / df_temp['этажей_до']
sample_rel = df_temp.sample(min(5000, len(df_temp)), random_state=42)
ax.scatter(sample_rel['относит_этаж'], sample_rel['цена_за_кв_м'], alpha=0.2, s=5)
ax.set_xlabel('Относительный этаж (этаж / макс)')
ax.set_ylabel('Цена за м² (₽)')
ax.set_title('Относительный этаж vs Цена')

plt.tight_layout()
plt.savefig('03_floor_analysis.png', dpi=150, bbox_inches='tight')
plt.close()

print("\nАнализ этажа:")
print(f"  Первый этаж дешевле на: {diff:,.0f} ₽/м²")
print(f"  Нижний этаж (с учётом этажей_от) дешевле на: {diff_lower:,.0f} ₽/м²")
print(f"  Последний этаж: {'+' if diff_last > 0 else ''}{diff_last:,.0f} ₽/м²")

# Проверка распределения этажей_от
print(f"\nРаспределение 'этажей_от':")
print(df['этажей_от'].value_counts().head(5))

# =============================================================================
# 6. АНАЛИЗ: УСТУПКА И ИПОТЕКА
# =============================================================================
print("\n" + "="*60)
print("6. АНАЛИЗ: УСТУПКА И ИПОТЕКА — влияют ли на цену за м²?")
print("="*60)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 6.1 Уступка
ax = axes[0]
df_temp = df.dropna(subset=['уступка', 'цена_за_кв_м'])
groups = df_temp.groupby('уступка')['цена_за_кв_м'].agg(['mean', 'std', 'count'])
ax.bar(range(len(groups)), groups['mean'], yerr=groups['std']/np.sqrt(groups['count'])*1.96, capsize=5)
ax.set_xticks(range(len(groups)))
ax.set_xticklabels(groups.index, rotation=45, ha='right')
ax.set_ylabel('Средняя цена за м² (₽)')
ax.set_title('Цена по типу уступки\n(влияет ли тип сделки на ЦЕНУ за м²?)')

# 6.2 Ипотека
ax = axes[1]
df_temp = df.dropna(subset=['ипотека', 'цена_за_кв_м'])
groups = df_temp.groupby('ипотека')['цена_за_кв_м'].agg(['mean', 'std', 'count'])
ax.bar(range(len(groups)), groups['mean'], yerr=groups['std']/np.sqrt(groups['count'])*1.96, capsize=5)
ax.set_xticks(range(len(groups)))
ax.set_xticklabels(groups.index, rotation=45, ha='right')
ax.set_ylabel('Средняя цена за м² (₽)')
ax.set_title('Цена по типу ипотеки')

# 6.3 Логическое обоснование
ax = axes[2]
ax.text(0.1, 0.9, "ВОПРОС: Должны ли уступка/ипотека влиять на цену за м²?", 
        fontsize=12, fontweight='bold', transform=ax.transAxes)
ax.text(0.1, 0.7, "• Уступка = покупка у первичного покупателя, а не застройщика", 
        fontsize=10, transform=ax.transAxes)
ax.text(0.1, 0.6, "• Ипотека = способ оплаты, не характеристика квартиры", 
        fontsize=10, transform=ax.transAxes)
ax.text(0.1, 0.4, "ВЫВОД:", fontsize=11, fontweight='bold', transform=ax.transAxes)
ax.text(0.1, 0.3, "Эти признаки описывают СДЕЛКУ, а не КВАРТИРУ.", 
        fontsize=10, transform=ax.transAxes)
ax.text(0.1, 0.2, "Их влияние может быть spurious (ложным).", 
        fontsize=10, transform=ax.transAxes)
ax.text(0.1, 0.1, "→ Рекомендация: ИСКЛЮЧИТЬ из модели", 
        fontsize=10, fontweight='bold', color='red', transform=ax.transAxes)
ax.axis('off')

plt.tight_layout()
plt.savefig('04_deal_type_analysis.png', dpi=150, bbox_inches='tight')
plt.close()

print("\n⚠️ ВЫВОД по уступке/ипотеке:")
print("   Эти признаки характеризуют СДЕЛКУ, а не КВАРТИРУ")
print("   Их корреляция с ценой может быть confounded другими факторами")
print("   → РЕШЕНИЕ: Исключить из финальной модели")

# =============================================================================
# 7. СОЗДАНИЕ ПРИЗНАКА МИНИМАЛЬНОЙ КОМНАТНОСТИ
# =============================================================================
print("\n" + "="*60)
print("7. СОЗДАНИЕ ПРИЗНАКА МИНИМАЛЬНОЙ КОМНАТНОСТИ")
print("="*60)

def get_min_rooms_flag(row):
    """
    Минимальная комнатность для класса:
    - Комфорт, Эконом: Студия
    - Бизнес-: 1-комн (студий нет)
    """
    if row['класс_жилья'] in ['Комфорт', 'Эконом']:
        return 1 if row['тип_квартиры'] == 'Студия' else 0
    elif row['класс_жилья'] == 'Бизнес-':
        return 1 if row['тип_квартиры'] == '1-комн' else 0
    return 0

def get_comparison_group(row):
    """
    Группы для сравнения:
    - мин: минимальная комнатность
    - средн: 2-3 комнатные (baseline)
    """
    if row['класс_жилья'] in ['Комфорт', 'Эконом']:
        if row['тип_квартиры'] == 'Студия':
            return 'мин'
        elif row['тип_квартиры'] in ['2-комн', '3-комн']:
            return 'средн'
        else:
            return 'другое'  # 1-комн - промежуточная категория
    elif row['класс_жилья'] == 'Бизнес-':
        if row['тип_квартиры'] == '1-комн':
            return 'мин'
        elif row['тип_квартиры'] in ['2-комн', '3-комн']:
            return 'средн'
        else:
            return 'другое'
    return 'другое'

df['мин_комнатность'] = df.apply(get_min_rooms_flag, axis=1)
df['группа_сравнения'] = df.apply(get_comparison_group, axis=1)

print("\nЛогика:")
print("  Комфорт/Эконом: Студия (мин) vs 2-3к (средн)")
print("  Бизнес-: 1-комн (мин) vs 2-3к (средн)")

print("\nРаспределение:")
print(pd.crosstab(df['класс_жилья'], df['группа_сравнения']))

# Средние цены
print("\nСредние цены за м² по группам:")
for cls in ['Комфорт', 'Эконом', 'Бизнес-']:
    cls_data = df[df['класс_жилья'] == cls]
    min_price = cls_data[cls_data['группа_сравнения'] == 'мин']['цена_за_кв_м'].mean()
    med_price = cls_data[cls_data['группа_сравнения'] == 'средн']['цена_за_кв_м'].mean()
    min_type = 'Студия' if cls in ['Комфорт', 'Эконом'] else '1-комн'
    diff = min_price - med_price
    diff_pct = (diff / med_price) * 100
    print(f"  {cls}: {min_type}={min_price:,.0f}, 2-3к={med_price:,.0f}, Δ={diff:+,.0f} ({diff_pct:+.1f}%)")

# =============================================================================
# 8. ПОДГОТОВКА ДАННЫХ ДЛЯ МОДЕЛИ
# =============================================================================
print("\n" + "="*60)
print("8. ПОДГОТОВКА ДАННЫХ ДЛЯ МОДЕЛИ")
print("="*60)

# Удаляем промежуточные категории (оставляем только мин vs средн)
df_model = df[df['группа_сравнения'].isin(['мин', 'средн'])].copy()
print(f"Записей для сравнения мин vs средн: {len(df_model):,}")

# Удаляем пропуски
required = ['цена_за_кв_м', 'этаж_лота', 'класс_жилья', 'район']
df_model = df_model.dropna(subset=required)
print(f"После удаления пропусков: {len(df_model):,}")

# Удаляем выбросы
q01, q99 = df_model['цена_за_кв_м'].quantile([0.01, 0.99])
df_model = df_model[(df_model['цена_за_кв_м'] >= q01) & (df_model['цена_за_кв_м'] <= q99)]
print(f"После удаления выбросов: {len(df_model):,}")

# Признаки этажа
# ВАЖНО: этажей_от - это НЕ "нижний жилой этаж", а минимальная этажность корпуса!
# Поэтому используем просто первый_этаж (=1)
df_model['первый_этаж'] = (df_model['этаж_лота'] == 1).astype(int)
df_model['последний_этаж'] = (df_model['этаж_лота'] == df_model['этажей_до']).astype(int)
df_model['относит_этаж'] = df_model['этаж_лота'] / df_model['этажей_до']

# Стадия
stage_mapping = {
    'Строительство не начато': 'Котлован',
    'Работы нулевого цикла': 'Котлован',
    'Начало монтажных работ': 'Строительство',
    'Монтажные и отделочные работы': 'Строительство',
    'Получение РВЭ, благоустройство территории': 'Почти готов',
    'Введен в эксплуатацию': 'Сдан'
}
df_model['стадия'] = df_model['текущая_стадия_строительства'].map(stage_mapping).fillna('Строительство')

# Отделка
def simplify_otdelka(x):
    if pd.isna(x): return 'Без отделки'
    x = str(x).lower()
    if 'с отделкой' in x and 'без' not in x: return 'С отделкой'
    elif 'под чистовую' in x: return 'Под чистовую'
    elif 'мебел' in x: return 'С мебелью'
    return 'Без отделки'

df_model['тип_отделки'] = df_model['внутренняя_отделка'].apply(simplify_otdelka)
df_model['есть_отделка'] = df_model['тип_отделки'].isin(['С отделкой', 'С мебелью', 'Под чистовую']).astype(int)

# Время
df_model['месяц_дата'] = pd.to_datetime(df_model['месяц'], errors='coerce')
min_date = df_model['месяц_дата'].min()
df_model['месяц_номер'] = ((df_model['месяц_дата'].dt.year - min_date.year) * 12 + 
                           (df_model['месяц_дата'].dt.month - min_date.month))

# Стандартизация
df_model['этаж_лота_std'] = (df_model['этаж_лота'] - df_model['этаж_лота'].mean()) / df_model['этаж_лота'].std()
df_model['месяц_номер_std'] = (df_model['месяц_номер'] - df_model['месяц_номер'].mean()) / df_model['месяц_номер'].std()

# =============================================================================
# 9. МОДЕЛЬ 1: МИНИМАЛИСТИЧНАЯ (только обоснованные признаки)
# =============================================================================
print("\n" + "="*60)
print("9. МОДЕЛЬ 1: МИНИМАЛИСТИЧНАЯ")
print("="*60)

print("""
Логика отбора признаков:
- мин_комнатность: ГЛАВНЫЙ признак (проверяем гипотезу)
- класс_жилья: влияет на качество и цену
- район: локация = главный фактор цены недвижимости
- стадия: готовность влияет на цену (риск vs готовое)
- есть_отделка: характеристика квартиры
- этаж_лота_std: характеристика квартиры (вид, шум)
- первый_этаж: дисконт за неудобства
- месяц_номер_std: временной тренд (инфляция, рынок)

ИСКЛЮЧЕНЫ:
- площадь: мультиколлинеарность с комнатностью
- застройщик: proxy для класса и района (confounding)
- уступка/ипотека: характеристика сделки, не квартиры
- последний_этаж: незначим в предыдущих моделях
""")

formula_minimal = """
цена_за_кв_м ~ 
    мин_комнатность +
    C(класс_жилья, Treatment(reference='Комфорт')) + 
    C(район, Treatment(reference='Коммунарка')) + 
    C(стадия, Treatment(reference='Строительство')) +
    есть_отделка +
    этаж_лота_std +
    первый_этаж +
    месяц_номер_std
"""

model_minimal = smf.ols(formula_minimal, data=df_model).fit()
model_minimal_robust = model_minimal.get_robustcov_results(cov_type='HC3')

print("\nМОДЕЛЬ 1 (минималистичная):")
print(model_minimal_robust.summary().as_text())

# VIF
X = model_minimal.model.exog
vif_data = []
for i, name in enumerate(model_minimal.model.exog_names):
    vif = variance_inflation_factor(X, i)
    vif_data.append({'Признак': name, 'VIF': vif})
vif_df = pd.DataFrame(vif_data).sort_values('VIF', ascending=False)
print("\nVIF (мультиколлинеарность):")
print(vif_df[vif_df['Признак'] != 'Intercept'].head(10).to_string(index=False))

# =============================================================================
# 10. МОДЕЛЬ 2: С ЗАСТРОЙЩИКОМ (для сравнения)
# =============================================================================
print("\n" + "="*60)
print("10. МОДЕЛЬ 2: С ЗАСТРОЙЩИКОМ (для сравнения)")
print("="*60)

# Топ застройщики
top_devs = df_model['застройщик'].value_counts().head(5).index.tolist()
df_model['застройщик_топ'] = df_model['застройщик'].apply(lambda x: x if x in top_devs else 'Другие')

formula_with_dev = """
цена_за_кв_м ~ 
    мин_комнатность +
    C(класс_жилья, Treatment(reference='Комфорт')) + 
    C(район, Treatment(reference='Коммунарка')) + 
    C(стадия, Treatment(reference='Строительство')) +
    есть_отделка +
    этаж_лота_std +
    первый_этаж +
    месяц_номер_std +
    C(застройщик_топ, Treatment(reference='Самолет'))
"""

model_with_dev = smf.ols(formula_with_dev, data=df_model).fit()
model_with_dev_robust = model_with_dev.get_robustcov_results(cov_type='HC3')

print("\nМОДЕЛЬ 2 (с застройщиком):")
print(f"R²: {model_with_dev_robust.rsquared:.4f} vs {model_minimal_robust.rsquared:.4f} (без застройщика)")
print(f"Δ R²: {(model_with_dev_robust.rsquared - model_minimal_robust.rsquared)*100:.2f}%")

# VIF с застройщиком
X2 = model_with_dev.model.exog
vif_data2 = []
for i, name in enumerate(model_with_dev.model.exog_names):
    vif = variance_inflation_factor(X2, i)
    vif_data2.append({'Признак': name, 'VIF': vif})
vif_df2 = pd.DataFrame(vif_data2).sort_values('VIF', ascending=False)
print("\nVIF (с застройщиком):")
print(vif_df2[vif_df2['Признак'] != 'Intercept'].head(10).to_string(index=False))

# =============================================================================
# 11. ОТДЕЛЬНЫЙ ТЕСТ ДЛЯ КОМФОРТ: Студии vs 4-комнатные
# =============================================================================
print("\n" + "="*60)
print("11. ОТДЕЛЬНЫЙ ТЕСТ: Студии vs 4-комнатные (только Комфорт)")
print("="*60)

# Используем исходный df (до фильтрации 4-комн)
df_comfort_full = deals.merge(
    projects.drop_duplicates(subset="id_корпуса", keep="first")[
        ["id_корпуса", "класс_проекта", "этажей_от", "этажей_до"]
    ],
    on="id_корпуса", how="left"
)
df_comfort_full['класс_жилья'] = df_comfort_full['класс'].fillna(df_comfort_full['класс_проекта'])
df_comfort_full['тип_квартиры'] = df_comfort_full['количество_комнат'].astype(str).apply(normalize_rooms)
df_comfort_full['кол_во_квартир'] = df_comfort_full['суммарное_количество_сделок'].fillna(1).astype(int)
df_comfort_full['площадь_квартиры'] = df_comfort_full['суммарная_площадь_сделок'] / df_comfort_full['кол_во_квартир']
df_comfort_full['реальная_цена_квартиры'] = df_comfort_full['реальная_сумма_бюджета'] / df_comfort_full['кол_во_квартир']
df_comfort_full['цена_за_кв_м'] = df_comfort_full['реальная_цена_квартиры'] / df_comfort_full['площадь_квартиры']

# Только Комфорт, студии и 4-комн
df_studio_4k = df_comfort_full[
    (df_comfort_full['класс_жилья'] == 'Комфорт') & 
    (df_comfort_full['тип_квартиры'].isin(['Студия', '4-комн']))
].dropna(subset=['цена_за_кв_м'])

# Удаляем выбросы
q01, q99 = df_studio_4k['цена_за_кв_м'].quantile([0.01, 0.99])
df_studio_4k = df_studio_4k[(df_studio_4k['цена_за_кв_м'] >= q01) & (df_studio_4k['цена_за_кв_м'] <= q99)]

studio_prices = df_studio_4k[df_studio_4k['тип_квартиры'] == 'Студия']['цена_за_кв_м']
fourroom_prices = df_studio_4k[df_studio_4k['тип_квартиры'] == '4-комн']['цена_за_кв_м']

print(f"\nРазмер выборки:")
print(f"  Студии: {len(studio_prices):,}")
print(f"  4-комнатные: {len(fourroom_prices):,}")

print(f"\nСредняя цена за м²:")
print(f"  Студии: {studio_prices.mean():,.0f} ₽")
print(f"  4-комнатные: {fourroom_prices.mean():,.0f} ₽")
print(f"  Разница: {studio_prices.mean() - fourroom_prices.mean():+,.0f} ₽ ({(studio_prices.mean()/fourroom_prices.mean()-1)*100:+.1f}%)")

# t-test
t_stat, p_value = stats.ttest_ind(studio_prices, fourroom_prices)
print(f"\nНезависимый t-тест:")
print(f"  t-статистика: {t_stat:.2f}")
print(f"  p-value: {p_value:.2e}")
print(f"  Результат: {'✅ Статистически значимая разница' if p_value < 0.05 else '⚪ Незначимо'}")

# Визуализация
fig, ax = plt.subplots(figsize=(10, 6))
ax.boxplot([studio_prices, fourroom_prices], labels=['Студия', '4-комнатная'])
ax.set_ylabel('Цена за м² (₽)')
ax.set_title(f'Комфорт: Студии vs 4-комнатные\nРазница: {studio_prices.mean() - fourroom_prices.mean():+,.0f} ₽/м² (p={p_value:.2e})')
plt.tight_layout()
plt.savefig('05_studio_vs_4room.png', dpi=150, bbox_inches='tight')
plt.close()

# =============================================================================
# 12. РЕЗУЛЬТАТЫ ГЛАВНОЙ МОДЕЛИ
# =============================================================================
print("\n" + "="*60)
print("12. РЕЗУЛЬТАТЫ ГЛАВНОЙ МОДЕЛИ")
print("="*60)

# Создаём DataFrame с результатами
feature_names = model_minimal.model.exog_names
conf_int = pd.DataFrame(model_minimal_robust.conf_int(), columns=['CI_025', 'CI_975'], index=feature_names)
results_df = pd.DataFrame({
    'Коэффициент': model_minimal_robust.params,
    'SE': model_minimal_robust.bse,
    't': model_minimal_robust.tvalues,
    'p_value': model_minimal_robust.pvalues
}, index=feature_names)
results_df = results_df.join(conf_int)

def stars(p):
    if p < 0.001: return '***'
    elif p < 0.01: return '**'
    elif p < 0.05: return '*'
    elif p < 0.1: return '.'
    return ''

results_df['Знач'] = results_df['p_value'].apply(stars)
results_df['|Коэф|'] = results_df['Коэффициент'].abs()
results_sorted = results_df.sort_values('|Коэф|', ascending=False)

print("\nКоэффициенты:")
for idx, row in results_sorted.iterrows():
    idx_str = str(idx)
    name = idx_str[:50] + '...' if len(idx_str) > 50 else idx_str
    print(f"{name:55} {row['Коэффициент']:>10,.0f}  p={row['p_value']:.2e} {row['Знач']:>3}")

# =============================================================================
# 13. ГЛАВНАЯ ГИПОТЕЗА
# =============================================================================
print("\n" + "="*80)
print("🎯 ГЛАВНАЯ ГИПОТЕЗА: Минимальная комнатность дороже за м²")
print("="*80)

coef = results_df.loc['мин_комнатность', 'Коэффициент']
pval = results_df.loc['мин_комнатность', 'p_value']
ci_low = results_df.loc['мин_комнатность', 'CI_025']
ci_high = results_df.loc['мин_комнатность', 'CI_975']

print(f"""
Сравнение:
  • Комфорт/Эконом: Студия vs 2-3 комнатные
  • Бизнес-: 1-комнатная vs 2-3 комнатные

Результат:
  Коэффициент: {coef:+,.0f} ₽/м²
  95% ДИ: [{ci_low:+,.0f}; {ci_high:+,.0f}]
  p-value: {pval:.2e}
  
  {'✅ ГИПОТЕЗА ПОДТВЕРЖДЕНА' if pval < 0.05 and coef > 0 else '❌ Не подтверждена'}
""")

# =============================================================================
# 14. ВИЗУАЛИЗАЦИИ
# =============================================================================
print("\n" + "="*60)
print("14. ВИЗУАЛИЗАЦИИ")
print("="*60)

# Коэффициенты
fig, ax = plt.subplots(figsize=(14, 10))
coef_plot = results_sorted[results_sorted.index != 'Intercept'].copy()

colors = ['gray' if p >= 0.05 else ('green' if c > 0 else 'red') 
          for c, p in zip(coef_plot['Коэффициент'], coef_plot['p_value'])]

y_pos = range(len(coef_plot))
ax.barh(y_pos, coef_plot['Коэффициент'], color=colors, alpha=0.8)
ax.errorbar(coef_plot['Коэффициент'], y_pos, 
           xerr=[coef_plot['Коэффициент'] - coef_plot['CI_025'], 
                 coef_plot['CI_975'] - coef_plot['Коэффициент']],
           fmt='none', color='black', capsize=3)
ax.set_yticks(y_pos)
ax.set_yticklabels([str(idx)[:40]+'...' if len(str(idx))>40 else str(idx) for idx in coef_plot.index], fontsize=9)
ax.axvline(x=0, color='black', linewidth=0.5)
ax.set_xlabel('Коэффициент (₽/м²)')
ax.set_title('Коэффициенты регрессии (минималистичная модель)\nзелёный=+, красный=−, серый=незначим', fontweight='bold')
plt.tight_layout()
plt.savefig('06_coefficients_minimal.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ 06_coefficients_minimal.png")

# Сравнение по классам
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, cls in enumerate(['Комфорт', 'Эконом', 'Бизнес-']):
    ax = axes[i]
    cls_data = df_model[df_model['класс_жилья'] == cls]
    
    min_type = 'Студия' if cls in ['Комфорт', 'Эконом'] else '1-комн'
    
    min_prices = cls_data[cls_data['группа_сравнения'] == 'мин']['цена_за_кв_м']
    med_prices = cls_data[cls_data['группа_сравнения'] == 'средн']['цена_за_кв_м']
    
    ax.boxplot([med_prices, min_prices], labels=['2-3к', min_type])
    ax.set_ylabel('Цена за м² (₽)')
    
    diff = min_prices.mean() - med_prices.mean()
    diff_pct = (diff / med_prices.mean()) * 100
    ax.set_title(f'{cls}\n{min_type}: {min_prices.mean():,.0f} vs 2-3к: {med_prices.mean():,.0f}\nΔ={diff:+,.0f} ({diff_pct:+.1f}%)')

plt.suptitle('Сравнение минимальной комнатности с 2-3 комнатными', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('07_min_vs_medium.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ 07_min_vs_medium.png")

# =============================================================================
# 15. ИТОГИ
# =============================================================================
print("\n" + "="*80)
print("📋 ИТОГОВЫЕ ВЫВОДЫ")
print("="*80)

errors = df_model['цена_за_кв_м'] - model_minimal.fittedvalues
rmse = np.sqrt(np.mean(errors**2))
mape = np.mean(np.abs(errors) / df_model['цена_за_кв_м']) * 100

print(f"""
┌─────────────────────────────────────────────────────────────────────────────────┐
│ КАЧЕСТВО МОДЕЛИ                                                                 │
├─────────────────────────────────────────────────────────────────────────────────┤
│   R²:          {model_minimal_robust.rsquared:.4f} (объясняет {model_minimal_robust.rsquared*100:.1f}% вариации)                 │
│   RMSE:        {rmse:,.0f} ₽                                                          │
│   MAPE:        {mape:.1f}%                                                             │
├─────────────────────────────────────────────────────────────────────────────────┤
│ ГЛАВНАЯ ГИПОТЕЗА                                                                │
├─────────────────────────────────────────────────────────────────────────────────┤
│   Минимальная комнатность дороже 2-3к на: {coef:+,.0f} ₽/м²                      │
│   95% ДИ: [{ci_low:+,.0f}; {ci_high:+,.0f}]                                                │
│   p-value: {pval:.2e}                                                             │
│   Результат: {'✅ ПОДТВЕРЖДЕНО' if pval < 0.05 and coef > 0 else '❌ НЕ подтверждено'}                                                      │
├─────────────────────────────────────────────────────────────────────────────────┤
│ ДОПОЛНИТЕЛЬНЫЙ ТЕСТ: Студии vs 4-комнатные (Комфорт)                            │
├─────────────────────────────────────────────────────────────────────────────────┤
│   Студии дороже 4к на: {studio_prices.mean() - fourroom_prices.mean():+,.0f} ₽/м²                                            │
│   p-value: {p_value:.2e}                                                             │
│   Результат: {'✅ ПОДТВЕРЖДЕНО' if p_value < 0.05 else '❌ НЕ подтверждено'}                                                      │
├─────────────────────────────────────────────────────────────────────────────────┤
│ КРИТИЧЕСКИЙ АНАЛИЗ ПРИЗНАКОВ                                                    │
├─────────────────────────────────────────────────────────────────────────────────┤
│   ✓ ИСКЛЮЧЕНА площадь (мультиколлинеарность с комнатностью, r={corr:.2f})         │
│   ✓ ИСКЛЮЧЁН застройщик (proxy для класса и района)                             │
│   ✓ ИСКЛЮЧЕНЫ уступка/ипотека (характеристика сделки, не квартиры)              │
│   ✓ Нижний этаж (с учётом этажей_от) вместо просто "первый этаж"                │
│   ✓ Max VIF < 5 (нет мультиколлинеарности)                                      │
└─────────────────────────────────────────────────────────────────────────────────┘
""")

# Сохранение
results_df.to_csv('regression_coefficients.csv')
df_model.to_csv('data_for_model.csv', index=False)

print("\n✓ Результаты сохранены в ")
print("\n" + "="*80)
print("✅ АНАЛИЗ ЗАВЕРШЁН")
print("="*80)

🔍 HEDONIC PRICING MODEL v4: КРИТИЧЕСКИЙ АНАЛИЗ ПРИЗНАКОВ

1. ЗАГРУЗКА ДАННЫХ
Загружено: 50,955 записей

2. ФИЛЬТРАЦИЯ ДАННЫХ
Исходно: 50,955
После удаления Бизнес: 50,948
После удаления 4-комн и 5+: 50,444
После удаления неуказанных: 50,444

Финальная структура:
тип_квартиры  1-комн  2-комн  3-комн  Студия
класс_жилья                                 
Бизнес-          424     394     198       0
Комфорт        19653   13375    5996    9380
Эконом           394     325     122     183

3. АНАЛИЗ: ПЛОЩАДЬ ↔ КОМНАТНОСТЬ (мультиколлинеарность)

✓ Корреляция площадь ↔ комнатность: r = 0.948
  ⚠️ ВЫСОКАЯ мультиколлинеарность! Нельзя включать оба признака.
  → РЕШЕНИЕ: Использовать только комнатность (тип_квартиры)
     Площадь уже заложена в комнатности.

4. АНАЛИЗ: ЗАСТРОЙЩИК — прямое влияние или proxy?

Анализ застройщиков:

А101:
  Классы: {'Комфорт': 1.0}
  Районы: {'Коммунарка': 0.88, 'Троицк': 0.09, 'Внуково': 0.03}

Самолет:
  Классы: {'Комфорт': 1.0}
  Районы: {'Щербинка': 0.37, 'Комм

In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

# Предполагаем, что df_model уже подготовлен

# 1.1 Простой график: средняя цена по месяцам (без контроля)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

ax = axes[0, 0]
monthly_raw = df_model.groupby('месяц_дата')['цена_за_кв_м'].mean()
ax.plot(monthly_raw.index, monthly_raw.values, marker='o', linewidth=2)
ax.set_xlabel('Месяц')
ax.set_ylabel('Средняя цена за м² (₽)')
ax.set_title('Сырые данные: средняя цена по месяцам\n(БЕЗ контроля других факторов)')
ax.tick_params(axis='x', rotation=45)

# 1.2 Проблема: распределение районов по месяцам
ax = axes[0, 1]
month_district = pd.crosstab(df_model['месяц_дата'], df_model['район'], normalize='index') * 100
month_district.plot(kind='area', stacked=True, ax=ax, alpha=0.7)
ax.set_xlabel('Месяц')
ax.set_ylabel('Доля района (%)')
ax.set_title('Доля районов по месяцам\n(если меняется → месяц confounded с районом)')
ax.legend(loc='upper left', fontsize=8)

# 1.3 Правильный способ: Fixed Effects по району
# Вычисляем "очищенную" цену внутри каждого района
df_model['цена_дцмм'] = df_model.groupby('район')['цена_за_кв_м'].transform(
    lambda x: x - x.mean()  # Демеанинг внутри района
)
monthly_demeaned = df_model.groupby('месяц_дата')['цена_дцмм'].mean()

ax = axes[1, 0]
ax.plot(monthly_demeaned.index, monthly_demeaned.values, marker='o', linewidth=2, color='green')
ax.axhline(y=0, color='gray', linestyle='--')
ax.set_xlabel('Месяц')
ax.set_ylabel('Отклонение от среднего района (₽)')
ax.set_title('С контролем района (demeaning)\n"Чистый" эффект времени')
ax.tick_params(axis='x', rotation=45)

# 1.4 Альтернатива: регрессия с dummy для месяцев
ax = axes[1, 1]
# Модель с месяцами как категориями
df_model['месяц_cat'] = df_model['месяц_дата'].dt.strftime('%Y-%m')
formula_month = "цена_за_кв_м ~ C(район) + C(месяц_cat)"
model_month = smf.ols(formula_month, data=df_model).fit()

# Извлекаем коэффициенты месяцев
month_coefs = {k: v for k, v in model_month.params.items() if 'месяц_cat' in k}
months = [k.split('[T.')[1].rstrip(']') for k in month_coefs.keys()]
values = list(month_coefs.values())

ax.bar(range(len(months)), values, color='steelblue')
ax.set_xticks(range(len(months)))
ax.set_xticklabels(months, rotation=45, ha='right', fontsize=8)
ax.axhline(y=0, color='gray', linestyle='--')
ax.set_ylabel('Коэффициент месяца (₽/м²)')
ax.set_title('Регрессия: эффект месяца\n(контроль района, baseline = первый месяц)')

plt.tight_layout()
plt.savefig('08_month_effect.png', dpi=150)
plt.close()

# Вывод статистики
print("Анализ влияния месяца:")
print(f"  Размах сырых средних: {monthly_raw.max() - monthly_raw.min():,.0f} ₽")
print(f"  Размах с контролем района: {monthly_demeaned.max() - monthly_demeaned.min():,.0f} ₽")
print(f"  Размах коэффициентов регрессии: {max(values) - min(values):,.0f} ₽")

Анализ влияния месяца:
  Размах сырых средних: 20,281 ₽
  Размах с контролем района: 17,718 ₽
  Размах коэффициентов регрессии: 17,927 ₽


In [24]:
# 2.1 Как вычисляется первый_этаж
print("="*60)
print("АНАЛИЗ: Этаж и первый_этаж")
print("="*60)

# Текущая логика
print("\nТекущая логика первого этажа:")
print("  первый_этаж = 1 если этаж_лота == 1, иначе 0")
print(f"  Количество первых этажей: {df_model['первый_этаж'].sum():,} из {len(df_model):,}")
print(f"  Доля: {df_model['первый_этаж'].mean()*100:.1f}%")

# 2.2 Проверка мультиколлинеарности
print("\n" + "="*60)
print("Мультиколлинеарность: этаж_лота_std vs первый_этаж")
print("="*60)

corr = df_model['этаж_лота_std'].corr(df_model['первый_этаж'])
print(f"  Корреляция: r = {corr:.3f}")

# Теоретическое объяснение
print("""
Теория:
  - первый_этаж = бинарный (0/1)
  - этаж_лота_std = непрерывный (стандартизированный)
  
  Корреляция будет УМЕРЕННОЙ (~ -0.3), потому что:
  - первый_этаж=1 → этаж_лота_std будет низким (но не обязательно минимальным)
  - первый_этаж=0 → этаж_лота_std может быть любым
  
  Это НЕ мультиколлинеарность, а дополняющие признаки:
  - этаж_лота_std: линейный тренд (выше этаж → дороже)
  - первый_этаж: дополнительный дисконт для 1-го этажа
""")

# 2.3 Визуализация
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Распределение этажей
ax = axes[0]
df_model['этаж_лота'].hist(bins=30, ax=ax, edgecolor='black')
ax.axvline(x=1, color='red', linewidth=2, label='Первый этаж')
ax.set_xlabel('Этаж')
ax.set_ylabel('Количество')
ax.set_title('Распределение этажей')
ax.legend()

# Цена по этажам
ax = axes[1]
floor_prices = df_model.groupby('этаж_лота')['цена_за_кв_м'].mean()
ax.plot(floor_prices.index, floor_prices.values, marker='o', markersize=3)
ax.axvline(x=1, color='red', linewidth=2, linestyle='--', label='Первый этаж')
ax.set_xlabel('Этаж')
ax.set_ylabel('Средняя цена за м² (₽)')
ax.set_title('Цена по этажам\n(виден провал на 1-м этаже)')
ax.legend()

# Проверка: первый этаж vs остальные низкие этажи
ax = axes[2]
df_model['категория_этажа'] = pd.cut(df_model['этаж_лота'], 
                                      bins=[0, 1, 3, 10, 100],
                                      labels=['1 (первый)', '2-3', '4-10', '11+'])
cat_prices = df_model.groupby('категория_этажа', observed=True)['цена_за_кв_м'].agg(['mean', 'std', 'count'])
ax.bar(range(len(cat_prices)), cat_prices['mean'], 
       yerr=cat_prices['std']/np.sqrt(cat_prices['count'])*1.96, capsize=5)
ax.set_xticks(range(len(cat_prices)))
ax.set_xticklabels(cat_prices.index)
ax.set_ylabel('Средняя цена за м² (₽)')
ax.set_title('Цена по категориям этажа\n(первый этаж — отдельный эффект)')

plt.tight_layout()
plt.savefig('09_floor_multicollinearity.png', dpi=150)
plt.close()

# 2.4 Альтернативные способы моделирования этажа
print("\n" + "="*60)
print("Альтернативные способы моделирования этажа:")
print("="*60)
print("""
1. Текущий (первый_этаж + этаж_std):
   ✓ Простой и интерпретируемый
   ✓ Ловит и линейный тренд, и особый эффект 1-го этажа
   
2. Относительный этаж (этаж / макс_этаж):
   ? Учитывает высоту дома
   - Сложнее интерпретировать
   
3. Категориальный (1, 2-3, 4-10, 11+):
   ✓ Гибкий, ловит нелинейности
   - Много dummy-переменных
   
4. Квадратичный (этаж + этаж²):
   ? Ловит убывающую предельную отдачу
   - Переусложнение для нашей задачи

Рекомендация: Текущий подход оптимален.
""")

АНАЛИЗ: Этаж и первый_этаж

Текущая логика первого этажа:
  первый_этаж = 1 если этаж_лота == 1, иначе 0
  Количество первых этажей: 319 из 29,018
  Доля: 1.1%

Мультиколлинеарность: этаж_лота_std vs первый_этаж
  Корреляция: r = -0.167

Теория:
  - первый_этаж = бинарный (0/1)
  - этаж_лота_std = непрерывный (стандартизированный)
  
  Корреляция будет УМЕРЕННОЙ (~ -0.3), потому что:
  - первый_этаж=1 → этаж_лота_std будет низким (но не обязательно минимальным)
  - первый_этаж=0 → этаж_лота_std может быть любым
  
  Это НЕ мультиколлинеарность, а дополняющие признаки:
  - этаж_лота_std: линейный тренд (выше этаж → дороже)
  - первый_этаж: дополнительный дисконт для 1-го этажа


Альтернативные способы моделирования этажа:

1. Текущий (первый_этаж + этаж_std):
   ✓ Простой и интерпретируемый
   ✓ Ловит и линейный тренд, и особый эффект 1-го этажа
   
2. Относительный этаж (этаж / макс_этаж):
   ? Учитывает высоту дома
   - Сложнее интерпретировать
   
3. Категориальный (1, 2-3, 4-10, 11+

In [27]:
from scipy import stats
from statsmodels.stats.diagnostic import het_breuschpagan, het_white
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

print("="*80)
print("ДИАГНОСТИКА МОДЕЛИ: Анализ остатков")
print("="*80)

# Получаем остатки
residuals = model_minimal.resid
fitted = model_minimal.fittedvalues
std_resid = (residuals - residuals.mean()) / residuals.std()

# =============================================
# 3.1 Нормальность остатков
# =============================================
print("\n1. НОРМАЛЬНОСТЬ ОСТАТКОВ")
print("-"*40)

# Тесты
shapiro_stat, shapiro_p = stats.shapiro(residuals.sample(min(5000, len(residuals)), random_state=42))
jb_stat, jb_p = stats.jarque_bera(residuals)
ks_stat, ks_p = stats.kstest(std_resid, 'norm')

print(f"  Shapiro-Wilk (n=5000): W={shapiro_stat:.4f}, p={shapiro_p:.2e}")
print(f"  Jarque-Bera: JB={jb_stat:.1f}, p={jb_p:.2e}")
print(f"  Kolmogorov-Smirnov: D={ks_stat:.4f}, p={ks_p:.2e}")
# print(f"  Skewness: {skew:.3f} (норма ~0)")
# print(f"  Kurtosis: {kurt:.3f} (норма ~3)")

if jb_p < 0.05:
    print("  ⚠️ Остатки не нормальны (p < 0.05)")
    print("     → Но при n>30,000 это часто не критично (ЦПТ)")
else:
    print("  ✓ Остатки нормальны")

# =============================================
# 3.2 Гетероскедастичность
# =============================================
print("\n2. ГЕТЕРОСКЕДАСТИЧНОСТЬ")
print("-"*40)

# Breusch-Pagan test
bp_stat, bp_p, _, _ = het_breuschpagan(residuals, model_minimal.model.exog)
print(f"  Breusch-Pagan: LM={bp_stat:.1f}, p={bp_p:.2e}")

# White test (если не слишком много переменных)
try:
    white_stat, white_p, _, _ = het_white(residuals, model_minimal.model.exog)
    print(f"  White: LM={white_stat:.1f}, p={white_p:.2e}")
except:
    print("  White: не удалось вычислить (много переменных)")

if bp_p < 0.05:
    print("  ⚠️ Гетероскедастичность обнаружена")
    print("     → Поэтому используем HC3 робастные ошибки")
else:
    print("  ✓ Гомоскедастичность")

# =============================================
# 3.3 Автокорреляция
# =============================================
print("\n3. АВТОКОРРЕЛЯЦИЯ")
print("-"*40)

dw = durbin_watson(residuals)
print(f"  Durbin-Watson: {dw:.3f}")
print(f"     Интерпретация: 2 = нет, <1.5 = положительная, >2.5 = отрицательная")

if dw < 1.5:
    print("  ⚠️ Возможна положительная автокорреляция")
    print("     → Для cross-sectional данных это может быть spatial autocorrelation")
elif dw > 2.5:
    print("  ⚠️ Возможна отрицательная автокорреляция")
else:
    print("  ✓ Автокорреляция не обнаружена")

# =============================================
# 3.4 Мультиколлинеарность (VIF)
# =============================================
print("\n4. МУЛЬТИКОЛЛИНЕАРНОСТЬ (VIF)")
print("-"*40)

X = model_minimal.model.exog
vif_data = []
for i, name in enumerate(model_minimal.model.exog_names):
    if name != 'Intercept':
        vif = variance_inflation_factor(X, i)
        vif_data.append({'Признак': name[:40], 'VIF': vif})

vif_df = pd.DataFrame(vif_data).sort_values('VIF', ascending=False)
print(vif_df.head(10).to_string(index=False))
print(f"\n  Max VIF: {vif_df['VIF'].max():.2f}")

if vif_df['VIF'].max() > 10:
    print("  ⚠️ Высокая мультиколлинеарность (VIF > 10)")
elif vif_df['VIF'].max() > 5:
    print("  ⚠️ Умеренная мультиколлинеарность (VIF > 5)")
else:
    print("  ✓ Мультиколлинеарность в норме (VIF < 5)")

# =============================================
# 3.5 Влиятельные наблюдения
# =============================================
print("\n5. ВЛИЯТЕЛЬНЫЕ НАБЛЮДЕНИЯ")
print("-"*40)

influence = model_minimal.get_influence()
cooks_d = influence.cooks_distance[0]
leverage = influence.hat_matrix_diag

# Пороги
n = len(residuals)
k = len(model_minimal.model.exog_names)
cooks_threshold = 4 / n
leverage_threshold = 2 * k / n

outliers_cooks = np.sum(cooks_d > cooks_threshold)
outliers_leverage = np.sum(leverage > leverage_threshold)

print(f"  Cook's distance > {cooks_threshold:.4f}: {outliers_cooks} ({outliers_cooks/n*100:.1f}%)")
print(f"  Leverage > {leverage_threshold:.4f}: {outliers_leverage} ({outliers_leverage/n*100:.1f}%)")

if outliers_cooks / n > 0.05:
    print("  ⚠️ Много влиятельных точек по Cook's D")
else:
    print("  ✓ Влиятельных точек немного")

# =============================================
# 3.6 Визуализация диагностики
# =============================================
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Residuals vs Fitted
ax = axes[0, 0]
ax.scatter(fitted, residuals, alpha=0.1, s=5)
ax.axhline(y=0, color='red', linestyle='--')
ax.set_xlabel('Предсказанные значения')
ax.set_ylabel('Остатки')
ax.set_title('Residuals vs Fitted\n(должны быть случайны вокруг 0)')

# Q-Q Plot
ax = axes[0, 1]
stats.probplot(std_resid, dist="norm", plot=ax)
ax.set_title('Q-Q Plot\n(должны лежать на диагонали)')

# Histogram of residuals
ax = axes[0, 2]
ax.hist(std_resid, bins=50, density=True, alpha=0.7, edgecolor='black')
x = np.linspace(-4, 4, 100)
ax.plot(x, stats.norm.pdf(x), 'r-', linewidth=2, label='Норм. распределение')
ax.set_xlabel('Стандартизированные остатки')
ax.set_ylabel('Плотность')
# ax.set_title(f'Распределение остатков\nSkew={skew:.2f}, Kurt={kurt:.2f}')
ax.legend()

# Scale-Location (sqrt of standardized residuals)
ax = axes[1, 0]
ax.scatter(fitted, np.sqrt(np.abs(std_resid)), alpha=0.1, s=5)
# Добавляем LOWESS
try:
    from statsmodels.nonparametric.smoothers_lowess import lowess
    smoothed = lowess(np.sqrt(np.abs(std_resid)), fitted, frac=0.1)
    ax.plot(smoothed[:, 0], smoothed[:, 1], 'r-', linewidth=2)
except:
    pass
ax.set_xlabel('Предсказанные значения')
ax.set_ylabel('√|Стандартизированные остатки|')
ax.set_title('Scale-Location\n(должна быть горизонтальная линия)')

# Residuals vs Leverage
ax = axes[1, 1]
ax.scatter(leverage, std_resid, alpha=0.1, s=5)
ax.axhline(y=0, color='red', linestyle='--')
ax.axvline(x=leverage_threshold, color='orange', linestyle='--', label=f'Порог={leverage_threshold:.4f}')
ax.set_xlabel('Leverage')
ax.set_ylabel('Стандартизированные остатки')
ax.set_title("Residuals vs Leverage\n(ищем точки с высоким leverage)")
ax.legend()

# Cook's distance
ax = axes[1, 2]
ax.stem(range(len(cooks_d)), cooks_d, markerfmt=',', basefmt=' ')
ax.axhline(y=cooks_threshold, color='red', linestyle='--', label=f'Порог={cooks_threshold:.4f}')
ax.set_xlabel('Индекс наблюдения')
ax.set_ylabel("Cook's Distance")
ax.set_title("Cook's Distance\n(влиятельные наблюдения)")
ax.legend()

plt.tight_layout()
plt.savefig('10_residual_diagnostics.png', dpi=150)
plt.close()

# =============================================
# 3.7 Итоговая таблица диагностики
# =============================================
print("\n" + "="*80)
print("ИТОГОВАЯ ДИАГНОСТИКА")
print("="*80)
print(f"""
┌─────────────────────────────────────────────────────────────────┐
│ Тест                    │ Статистика    │ p-value  │ Результат │
├─────────────────────────────────────────────────────────────────┤
│ Jarque-Bera (нормальн.) │ {jb_stat:>10.1f}    │ {jb_p:.2e} │ {'⚠️' if jb_p < 0.05 else '✓'}        │
│ Breusch-Pagan (гетеро.) │ {bp_stat:>10.1f}    │ {bp_p:.2e} │ {'⚠️ → HC3' if bp_p < 0.05 else '✓'}   │
│ Durbin-Watson (автокор.)│ {dw:>10.3f}    │    —     │ {'⚠️' if dw < 1.5 else '✓'}        │
│ Max VIF (мультиколл.)   │ {vif_df['VIF'].max():>10.2f}    │    —     │ {'⚠️' if vif_df['VIF'].max() > 5 else '✓'}        │
│ Cook's D outliers       │ {outliers_cooks/n*100:>9.1f}%    │    —     │ {'⚠️' if outliers_cooks/n > 0.05 else '✓'}        │
└─────────────────────────────────────────────────────────────────┘

Рекомендации:
- Гетероскедастичность → используем HC3 робастные ошибки ✓
- Ненормальность остатков → при n>{n:,} это не критично (ЦПТ) ✓
- Низкий Durbin-Watson → возможна пространственная автокорреляция
  (можно добавить кластеризацию по проектам или районам)
""")

ДИАГНОСТИКА МОДЕЛИ: Анализ остатков

1. НОРМАЛЬНОСТЬ ОСТАТКОВ
----------------------------------------
  Shapiro-Wilk (n=5000): W=0.9986, p=2.06e-04
  Jarque-Bera: JB=86.0, p=2.13e-19
  Kolmogorov-Smirnov: D=0.0137, p=3.47e-05
  ⚠️ Остатки не нормальны (p < 0.05)
     → Но при n>30,000 это часто не критично (ЦПТ)

2. ГЕТЕРОСКЕДАСТИЧНОСТЬ
----------------------------------------
  Breusch-Pagan: LM=2307.0, p=0.00e+00
  White: LM=3241.8, p=0.00e+00
  ⚠️ Гетероскедастичность обнаружена
     → Поэтому используем HC3 робастные ошибки

3. АВТОКОРРЕЛЯЦИЯ
----------------------------------------
  Durbin-Watson: 0.539
     Интерпретация: 2 = нет, <1.5 = положительная, >2.5 = отрицательная
  ⚠️ Возможна положительная автокорреляция
     → Для cross-sectional данных это может быть spatial autocorrelation

4. МУЛЬТИКОЛЛИНЕАРНОСТЬ (VIF)
----------------------------------------
                                 Признак      VIF
C(район, Treatment(reference='Коммунарка 1.421961
C(класс_жилья, Treatme